# 02 — Context Isolation

**CCA Pattern**: Subagents do NOT inherit coordinator context. Each starts blank — it receives ONLY what was explicitly passed.

This is the concept that trips up more candidates than any other.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
sys.path.insert(0, str(Path('.').resolve()))

In [ ]:
from research_agents.models.research import SubTask
from research_agents.agent.context_builder import build_subagent_context
from research_agents.agent.agent_loop import AgentResult

## Anti-Pattern: Shared Context

The coordinator passes its full message history to the subagent.

In [ ]:
from research_agents.anti_patterns.shared_context import run_leaky_subagent

# Simulate coordinator messages
coordinator_messages = [
    {'role': 'user', 'content': 'Research renewable energy globally'},
    {'role': 'assistant', 'content': 'I will decompose this into subtasks...'},
    {'role': 'user', 'content': 'Use APA citation format for all sources'},
]

# The leaky function converts messages to str() — wasteful and polluting
leaked_context = str(coordinator_messages)
print(f'Leaked context length: {len(leaked_context)} chars')
print(f'Contains APA instruction: {"APA" in leaked_context}')
print(f'Contains coordinator reasoning: {"decompose" in leaked_context}')

## Correct Pattern: Explicit Context Passing

The context builder passes ONLY what the coordinator deliberately selects.

In [ ]:
# Correct: SubTask contains only explicit context
task = SubTask(
    task_id='t1',
    agent_type='web_researcher',
    instruction='Search for renewable energy adoption statistics',
    context='Focus on 2024 data from government and peer-reviewed sources. Use APA citation format.',
)

explicit_context = build_subagent_context(task)
print(f'Explicit context length: {len(explicit_context)} chars')
print(f'Contains instruction: {"renewable energy" in explicit_context}')
print(f'Contains APA format: {"APA" in explicit_context}')
print(f'Contains coordinator reasoning: {"decompose" in explicit_context}')
print()
print(explicit_context)

In [ ]:
from helpers import compare_results

compare_results(
    {'context_length': len(leaked_context), 'contains_coordinator_reasoning': True, 'contains_other_agent_results': True, 'type': 'str(messages)'},
    {'context_length': len(explicit_context), 'contains_coordinator_reasoning': False, 'contains_other_agent_results': False, 'type': 'explicit_string'},
)

## CCA Exam Tip

> Any question where a subagent produces results that 'should have followed the coordinator\'s instructions' is testing context isolation.
> - The answer is always that the instructions were in the coordinator's context but never explicitly forwarded
> - Subagents do not inherit. Subagents receive only what you explicitly send.